## Visualização dos Dados

Para as visualizações, apenas os eletrodos de interesse são considerados. Os eletrodos EEG são os mesmos utilizados por Nieto et al. (2022) na análise espectral (p. 20, Figura 9). Os eletrodos EXG são os 8 canais externos de referência (EOG e EMG).

In [ ]:
# Imports
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add the Inner Speech Dataset repository to the system path
sys.path.append(str(Path.cwd().parent / "src" / "Python_Processing"))

# List with every subject (S1-S10)
N_S = list(range(1, 11))

# sub-09 ses-01 and sub-10 ses-03 are truncated in the OpenNeuro datasets
EXCLUDE = {9, 10}
SUBJECTS = [i for i in N_S if i not in EXCLUDE]
CLASSES = {0: "up", 1: "down", 2: "right", 3: "left"}

# EEG electrodes used by Nieto et al.
EEG_ELECTRODES = {
    "A": [4, 5, 10, 13, 17, 19, 20, 21, 22, 26, 30, 32],
    "B": [7, 16, 22, 24, 29],
    "D": [10, 19, 21, 26],
}

# Directory with the np arrays generated in 02-load
interim_dir = Path.cwd().parent / "data" / "interim"
processed_dir = Path.cwd().parent / "data" / "processed"

# Directory where figures are saved
fig_dir = Path.cwd().parent / "figures"
fig_dir.mkdir(exist_ok=True)

In [ ]:

# Channel names of interest ("A4", "A5", "A10", ..., "D26" and "EXG1", "EXG2", ..., "EXG8"
ch_eeg = [f"{letter}{n}" for letter, nums in EEG_ELECTRODES.items() for n in nums]
ch_exg = [f"EXG{i}" for i in range(1, 9)]

# Channels and time points are the same for every subject
names_eeg = np.load(interim_dir / "S1_eeg_channels.npy")
names_exg = np.load(interim_dir / "S1_exg_channels.npy")
times = np.load(interim_dir / "S1_times.npy")

# Data in (classes, trials, channels, time points)
X_eeg = np.load(processed_dir / "X_eeg.npy", mmap_mode="r")
X_exg = np.load(processed_dir / "X_exg.npy", mmap_mode="r")

# subj[c, j] = subject that trial j of class c came from
subj = np.load(processed_dir / "subj.npy")

# Position of each channel of interest in the channel axis of X_eeg and X_exg
idx_eeg = [list(names_eeg).index(c) for c in ch_eeg]
idx_exg = [list(names_exg).index(c) for c in ch_exg]

In [ ]:
# Plots the mean signal pf every channel in ch_idx
def plot_channels(X, names, ch_idx, classes=(0, 1, 2, 3), subject=1, 
                  n_cols=7, title="Mean EEG", save_as=None):    
    # One subplot per channel in a grid with n_cols columns
    n_ch = len(ch_idx)
    n_rows = int(np.ceil(n_ch / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 2.0 * n_rows),
                             sharex=True, sharey=True, squeeze=False)

    for c in classes:
        # Keep only the trials of the chosen subject
        idx_subj = (subj[c] == subject)
        trials = X[c][idx_subj][:, ch_idx] # trials x selected channels x time ppints
        mean = trials.mean(axis=0) * 1e6 # multiplied by 1e6 to convert volts to microvolts

        # Draw each class' mean in each channel's subplot
        for k, ax in enumerate(axes.flat[:n_ch]):
            ax.plot(times, mean[k], lw=0.5, label=CLASSES[c])

    # Define channel name as title, and a vertical line at t = 0
    for k, ax in enumerate(axes.flat[:n_ch]):
        ax.set_title(str(names[ch_idx[k]]), fontsize=8, pad=2)
        ax.tick_params(labelsize=6)
        ax.axvline(0, color="black", lw=0.4, alpha=0.4)

    sbj =f"S{subject}"
    fig.legend(labels=[CLASSES[c] for c in classes], loc="upper center",
               bbox_to_anchor=(0.5, 1.0), ncol=len(classes), fontsize=10, frameon=False)
    fig.suptitle(f"{title} for {sbj}, volts (µV) vs. time (s)", y=1.002)
    fig.tight_layout()

    if save_as:
        fig.savefig(save_as, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# For each subject, plot the EEG and EXG electrodes of interest
for i in SUBJECTS:
    plot_channels(X_eeg, names_eeg, idx_eeg, subject=i, n_cols=7,
                  title="Mean EEG, only electrodes of interest",
                  save_as=fig_dir / f"s{i}_eeg.png")
    
    plot_channels(X_exg, names_exg, idx_exg, subject=i, n_cols=4,
                  title="Mean EXG",
                  save_as=fig_dir / f"s{i}_exg.png")

In [ ]:
# Debugging
X_eeg = np.load(processed_dir / "X_eeg.npy")
subj = np.load(processed_dir / "subj.npy")
ses = np.load(processed_dir / "ses.npy")
X_exg = np.load(processed_dir / "X_exg.npy")

print(X_eeg.shape, X_exg.shape, subj.shape, ses.shape)
print(np.unique(subj[0], return_counts=True)) # trials per subject in class 0
print(np.unique(ses, return_counts=True)) # sessions 1, 2, 3 only
print((subj == subj[0]).all()) # same subject order in every class